In [43]:
import numpy as np
from pathlib import Path

from scipy.fft import fft, ifft, dct, idct, dst, idst

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import StatePreparation, HGate, SwapGate, MCXGate, SdgGate
from qiskit.quantum_info import Statevector

from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel

## Configs

In [44]:
WINDOW_SIZE = 256

RETENTION_RATIOS = [1.00, 0.75, 0.50, 0.25, 0.10, 0.05, 0.02, 0.01]

SEED = 42

CW_DATASET = Path("cw_dataset")
CLEAN_DIR = CW_DATASET / "clean"
NOISY_DIR = CW_DATASET / "noisy"
ENVELOPE_DIR = CW_DATASET / "envelopes"
AUDIO_DIR = CW_DATASET / "audio"
AUDIO_CLEAN_DIR = AUDIO_DIR / "clean"
AUDIO_NOISY_DIR = AUDIO_DIR / "noisy"

## Helper functions

In [45]:
def validate_window(x):
    x = np.asarray(x)
    if x.ndim != 1:
        raise ValueError("Input must be 1-D")

    n = len(x)
    if n == 0 or (n & (n - 1)) != 0:
        raise ValueError(f"Length must be a power of two, got {n}")

    return x

def prepare_amplitudes(x):
    x = validate_window(x)
    norm = np.linalg.norm(x)
    if norm <= 1e-15:
        return None, 0.0

    amplitudes = (x / norm).astype(np.complex128)
    num_qubits = int(np.log2(len(x)))
    qc = QuantumCircuit(num_qubits)
    qc.append(StatePreparation(amplitudes), range(num_qubits),)
    return qc, norm

def run_transform(x, transform_circuit):
    prep, input_norm = prepare_amplitudes(x)
    if prep is None:
        return (np.zeros_like(x, dtype=np.complex128,), None,)

    if prep.num_qubits != transform_circuit.num_qubits:
        raise ValueError("State preparation and transform have different qubit counts.")

    prep.compose(transform_circuit, inplace=True,)
    state = Statevector.from_instruction(prep).data
    result = (np.asarray(state) * input_norm)
    return result, prep

def append_zero_controlled_h(qc, controls, target):
    if not controls:
        qc.h(target)
        return

    for q in controls:
        qc.x(q)

    gate = HGate().control(len(controls))
    qc.append(gate, [*controls, target])
    for q in controls:
        qc.x(q)

def append_zero_controlled_swap(qc, controls, q1, q2):
    if not controls:
        qc.swap(q1, q2)
        return

    for q in controls:
        qc.x(q)

    gate = SwapGate().control(len(controls))
    qc.append(gate, [*controls, q1, q2])
    for q in controls:
        qc.x(q)

def append_mcx(qc, controls, target):
    controls = list(controls)
    if len(controls) == 0:
        qc.x(target)
    elif len(controls) == 1:
        qc.cx(controls[0], target)
    else:
        qc.append(MCXGate(len(controls)), [*controls, target])

def append_ctrl_ones_complement(qc, data, ctrl):
    for q in data:
        qc.cx(ctrl, q)

def append_ctrl_increment(qc, data, ctrl):
    n = len(data)
    for i in range(n - 1, 0, -1):
        controls = [ctrl, *data[:i]]
        append_mcx(qc, controls, data[i])
    qc.cx(ctrl, data[0],)

def append_ctrl_decrement(qc, data, ctrl):
    n = len(data)
    qc.cx(ctrl, data[0],)
    for i in range(1, n):
        controls = [ctrl, *data[:i],]
        append_mcx(qc, controls, data[i])

def append_ctrl_twos_complement(qc, data, ctrl):
    append_ctrl_ones_complement(qc, data, ctrl)
    append_ctrl_increment(qc, data, ctrl)

def append_vn(qc, data, ctrl):
    # H ⊗ I
    qc.h(ctrl)
    # pi_1:
    # |1,x> -> |1, one's_complement(x)>
    append_ctrl_ones_complement(qc, data, ctrl)

def append_d1(qc, data, ctrl):
    n = len(data)
    N = 2 ** n
    theta = np.pi / (2 * N)
    # delta_2:
    # active when ctrl = 1
    # K_i = X L_i^dag X
    for i, q in enumerate(data):
        angle = (2 ** i) * theta
        qc.x(q)
        qc.cp(-angle, ctrl, q)
        qc.x(q)

    # delta_1:
    # active when ctrl = 0
    qc.x(ctrl)
    for i, q in enumerate(data):
        angle = (2 ** i) * theta
        qc.cp(angle, ctrl, q)

    qc.x(ctrl)
    # C = diag(1, conjugate(omega))
    qc.p(-theta, ctrl,)

def append_g(qc, data, ctrl):
    # Base B^T = S H
    qc.h(ctrl)
    qc.s(ctrl)
    # Convert condition:
    # data == 000...0
    # into
    # data == 111...1
    for q in data:
        qc.x(q)

    controls = list(data)
    # Controlled J = Sdg H Sdg
    qc.append(SdgGate().control(len(controls)), [*controls, ctrl])
    qc.append(HGate().control(len(controls)), [*controls, ctrl])
    qc.append(SdgGate().control(len(controls)), [*controls, ctrl])
    # Restore data
    for q in data:
        qc.x(q)

def append_un_dagger(qc, data, ctrl):
    # D1
    append_d1(qc, data, ctrl)
    # Conditional two's complement
    append_ctrl_twos_complement(qc, data, ctrl)
    # Conditional branch mixing
    append_g(qc, data, ctrl,)
    # Conditional decrement
    append_ctrl_decrement(qc, data, ctrl)

def run_qdct_transform(x, transform):
    x = validate_window(x)
    norm = np.linalg.norm(x)
    if norm <= 1e-15:
        return (np.zeros_like(x, dtype=np.complex128,), None, 0.0,)

    N = len(x)
    n = int(np.log2(N))
    if transform.num_qubits != n + 1:
        raise ValueError("QDCT circuit must contain " "n data qubits + 1 control qubit")

    qc = QuantumCircuit(n + 1)
    amplitudes = (x / norm).astype(np.complex128)
    # Prepare only data.
    # ctrl remains |0>.
    qc.append(StatePreparation(amplitudes), range(n),)
    qc.compose(transform, qubits=range(n + 1), inplace=True,)
    state = np.asarray(Statevector.from_instruction(qc).data)
    # ctrl is the MSB.
    # ctrl = 0 occupies state[0:N].
    cosine_branch = state[:N]
    coeffs = (cosine_branch * norm)
    cosine_probability = np.sum(np.abs(cosine_branch) ** 2)
    leakage = (1.0 - cosine_probability)
    return (coeffs, qc, leakage)

## FFT, IFFT

In [46]:
def classical_fft(x):
    x = validate_window(x)
    return fft(x, norm="ortho")

def classical_ifft(coeffs):
    coeffs = validate_window(coeffs)
    return ifft(coeffs, norm="ortho")

## QFT, IQFT

In [47]:
def build_qft(num_qubits):
    qc = QuantumCircuit(num_qubits, name="QFT")
    for target in reversed(range(num_qubits)):
        qc.h(target)
        for control in reversed(range(target)):
            distance = target - control
            angle = np.pi / (2 ** distance)
            qc.cp(angle, control, target)

    for q in range(num_qubits // 2):
        qc.swap(q, num_qubits - q - 1)

    return qc

def build_iqft(num_qubits):
    qc = build_qft(num_qubits).inverse()
    qc.name = "IQFT"
    return qc

## QFFT, IQFFT

In [48]:
def quantum_fft(x):
    x = validate_window(x)
    num_qubits = int(np.log2(len(x)))
    transform = build_iqft(num_qubits)
    return run_transform(x, transform)

def quantum_ifft(coeffs):
    coeffs = validate_window(coeffs)
    num_qubits = int(np.log2(len(coeffs)))
    transform = build_qft(num_qubits)
    return run_transform(coeffs, transform)

## DWT Haar

In [49]:
def classical_haar(x):
    x = validate_window(x).astype(np.float64)
    approx = x.copy()
    details = []
    while len(approx) > 1:
        even = approx[0::2]
        odd = approx[1::2]
        next_approx = (even + odd) / np.sqrt(2.0)
        detail = (even - odd) / np.sqrt(2.0)
        details.append(detail)
        approx = next_approx

    return np.concatenate([approx, *details[::-1]])

def classical_ihaar(coeffs):
    coeffs = validate_window(coeffs).astype(np.float64)
    approx = coeffs[:1]
    offset = 1
    while offset < len(coeffs):
        n = len(approx)
        detail = coeffs[offset:offset + n]
        reconstructed = np.empty(2 * n, dtype=np.float64)
        reconstructed[0::2] = (approx + detail) / np.sqrt(2.0)
        reconstructed[1::2] = (approx - detail) / np.sqrt(2.0)
        approx = reconstructed
        offset += n

    return approx

## QDWT Haar

In [50]:
def build_qhaar(num_qubits):
    qc = QuantumCircuit(num_qubits, name="QDWT-Haar")

    for level in range(num_qubits):
        active_qubits = (num_qubits - level)

        controls = list(range(active_qubits, num_qubits))

        # Haar average/detail pair
        append_zero_controlled_h(qc, controls, target=0)

        # Pack the new detail bit at the
        # upper edge of the active region.
        # q0 -> q(active_qubits - 1)
        if active_qubits > 1:
            for q in range(active_qubits - 1):
                append_zero_controlled_swap(qc, controls, q, q + 1,)
    return qc

def build_iqhaar(num_qubits):
    qc = build_qhaar(num_qubits).inverse()
    qc.name = "IQDWT-Haar"
    return qc

def quantum_haar(x):
    x = validate_window(x)
    num_qubits = int(np.log2(len(x)))
    transform = build_qhaar(num_qubits)
    return run_transform(x, transform,)

def quantum_ihaar(coeffs):
    coeffs = validate_window(coeffs)
    num_qubits = int(np.log2(len(coeffs)))
    transform = build_iqhaar(num_qubits)
    return run_transform(coeffs, transform,)

## DCT-II

In [51]:
def classical_dct(x):
    x = validate_window(x)
    return dct(x, type=2, norm="ortho")

def classical_idct(coeffs):
    coeffs = validate_window(coeffs)
    return idct(coeffs, type=2, norm="ortho")

## QDCT-II

In [52]:
def build_qdct(num_data_qubits):
    num_total_qubits = (num_data_qubits + 1)
    qc = QuantumCircuit(num_total_qubits, name="QDCT-II")
    data = list(range(num_data_qubits))
    ctrl = num_data_qubits
    # 1. V_N
    append_vn(qc, data, ctrl)
    # 2. F_{2N}
    # IMPORTANT:
    # Here we need the mathematical positive-sign QFT,
    # NOT quantum_fft()/IQFT.
    qft = build_qft(num_total_qubits)
    qc.compose(qft, qubits=range(num_total_qubits), inplace=True)
    # 3. U_N^\dagger
    append_un_dagger(qc, data, ctrl,)
    return qc

def build_iqdct(num_data_qubits):
    qc = build_qdct(num_data_qubits).inverse()
    qc.name = "IQDCT"
    return qc

def quantum_dct(x):
    x = validate_window(x)
    n = int(np.log2(len(x)))
    transform = build_qdct(n)
    return run_qdct_transform(x, transform)

def quantum_idct(coeffs):
    coeffs = validate_window(coeffs)
    n = int(np.log2(len(coeffs)))
    transform = build_iqdct(n)
    return run_qdct_transform(coeffs, transform)

## Sanity checks

In [53]:
rng = np.random.default_rng(42)
x = rng.normal(size=8).astype(np.float64)
pairs = [
    ("FFT/QFT",   classical_fft, classical_ifft, quantum_fft,  quantum_ifft),
    ("DCT/QDCT", classical_dct, classical_idct, quantum_dct, quantum_idct),
    # ("DST/QDST", classical_dst, classical_idst, quantum_dst, quantum_idst),
    ("Haar/QDWT", classical_haar, classical_ihaar, quantum_haar, quantum_ihaar),
]

for name, classical_fn, classical_ifn, quantum_fn, quantum_inverse_fn in pairs:
    classical_result = classical_fn(x)
    quantum_result, _, _ = quantum_fn(x)
    quantum_reconstructed, _ = quantum_inverse_fn(quantum_result)
    error = np.max(np.abs(classical_result - quantum_result))
    reconstruction_error = np.max(np.abs(x - quantum_reconstructed))
    classical_reconstructed = classical_ifn(classical_result)
    classical_reconstruction_error = np.max(np.abs(x - classical_reconstructed))
    print(f"{name:12s} | transform error = {error:.6e} |  quantum reconstruction error = {reconstruction_error:.6e} | classical reconstruction error = {classical_reconstruction_error:.6e}")
    print("Input:")
    print(np.round(x, 4))
    print("Classical:")
    print(np.round(classical_result, 4))
    print("Quantum_real:")
    print(np.round(quantum_result.real, 4))
    print("Quantum:")
    print(np.round(quantum_result, 4))

ValueError: not enough values to unpack (expected 3, got 2)